# 🦅 FALCON CFE — Entrenamiento de IA satelital (con Google Drive)

Entrena el detector de infraestructura de CFE con **GPU gratis**, guardando todo en tu **Google Drive** para que **NO pierdas el progreso si Colab se desconecta**.

**Activa la GPU primero:** Menú *Entorno de ejecución → Cambiar tipo* → **GPU (T4)** → Guardar.

Corre las celdas en orden. Si se desconecta a media faena, ve a la sección **REANUDAR** al final.

## 1. Verificar GPU

In [ ]:
!nvidia-smi

## 2. Conectar Google Drive (aquí se guarda todo)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
BASE = '/content/drive/MyDrive/falcon_cfe'
os.makedirs(BASE, exist_ok=True)
print('Todo se guardara en:', BASE)

## 3. Clonar el repositorio e instalar dependencias

In [ ]:
!git clone https://github.com/Prime119/ProyectoMEC.git
%cd ProyectoMEC
!pip install -q pillow httpx ultralytics

## 4. Preparar dataset en Drive (hasta 2000 imágenes auto-etiquetadas)

Se guarda en Drive, así que **solo lo haces una vez**. Si ya lo generaste antes, puedes saltar esta celda.

In [ ]:
!python entrenamiento/preparar_dataset.py --zoom 18 --radio 200 --max-imagenes 2000 --salida /content/drive/MyDrive/falcon_cfe/datos

## 5. Generar configuración (apuntando al dataset en Drive)

In [ ]:
!python entrenamiento/generar_config.py --ruta /content/drive/MyDrive/falcon_cfe/datos

## 6. Entrenar (guarda checkpoints en Drive)

Con GPU T4 y ~2000 imágenes, 100 épocas tardan ~1-2 h. Los pesos se guardan en Drive cada 10 épocas.

In [ ]:
!python entrenamiento/entrenar.py --modelo yolov8m.pt --epochs 100 --imgsz 960 --device 0 --project /content/drive/MyDrive/falcon_cfe/runs

## 7. El modelo ya está en tu Drive

Al terminar, el modelo queda en `MyDrive/falcon_cfe/runs/cfe_satelital/weights/best.onnx`.
Descárgalo también a tu PC con la siguiente celda:

In [ ]:
import glob, shutil
from google.colab import files
onnx = glob.glob('/content/drive/MyDrive/falcon_cfe/runs/**/best.onnx', recursive=True)
if onnx:
    print('Modelo:', onnx[0])
    shutil.copy(onnx[0], 'cfe_satelital.onnx')
    files.download('cfe_satelital.onnx')
else:
    print('Aun no existe best.onnx; revisa que el entrenamiento haya terminado.')

---
## 🔄 REANUDAR (si Colab se desconectó a media faena)

No pierdes nada: el dataset y los checkpoints están en Drive. Solo corre EN ORDEN:
1. La celda **1** (GPU), **2** (Drive) y **3** (clonar + instalar)
2. La celda **5** (generar config apuntando a Drive)
3. Y luego la celda de aquí abajo (reanuda desde el último checkpoint):

In [ ]:
!python entrenamiento/entrenar.py --nombre cfe_satelital --project /content/drive/MyDrive/falcon_cfe/runs --resume

## (Opcional) Ver métricas y ejemplos de detección

In [ ]:
import glob
from IPython.display import Image, display
for img in glob.glob('/content/drive/MyDrive/falcon_cfe/runs/**/results.png', recursive=True):
    display(Image(filename=img))
for img in glob.glob('/content/drive/MyDrive/falcon_cfe/runs/**/val_batch0_pred.jpg', recursive=True)[:2]:
    display(Image(filename=img))